# 01 Verify Cleaned Data

This notebook verifies that the Silver layer is usable for feature engineering after applying the Bronze-informed cleaning policy.

The notebook intentionally avoids loading full telemetry tables into memory. Telemetry row counts and schemas come from Parquet metadata, while detailed profiling is limited to operational tables.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "01_verify_cleaned_data"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 01_verify_cleaned_data
Start time: 2026-06-02 00:45:50.034288
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


## File Coverage

Silver should contain one Parquet artifact per active endpoint. This check verifies file presence, row counts, and storage footprint without treating telemetry file size as a quality signal.

In [2]:
expected_files = [
    "meetings.parquet",
    "sessions.parquet",
    "drivers.parquet",
    "session_result.parquet",
    "laps.parquet",
    "weather.parquet",
    "stints.parquet",
    "starting_grid.parquet",
    "intervals.parquet",
    "position.parquet",
    "overtakes.parquet",
    "car_data.parquet",
    "location.parquet",
]
records = []
for name in expected_files:
    path = CLEANED_DATA_PATH / name
    if not path.exists():
        records.append({"file": name, "exists": False, "rows": 0, "columns": 0, "size_mb": 0.0, "endpoint_group": "Missing"})
        continue
    parquet_file = pq.ParquetFile(path)
    records.append({
        "file": name,
        "exists": True,
        "rows": int(parquet_file.metadata.num_rows),
        "columns": len(parquet_file.schema.names),
        "size_mb": path.stat().st_size / 1_000_000,
        "endpoint_group": "Telemetry" if name in {"car_data.parquet", "location.parquet"} else "Operational",
    })
file_df = pd.DataFrame(records)
file_df.to_csv(OUTPUT_TABLES / "file_inventory.csv", index=False)
display(file_df)

,file,exists,rows,columns,size_mb,endpoint_group
0,meetings.parquet,True,76,18,0.017615,Operational
1,sessions.parquet,True,70,15,0.012342,Operational
2,drivers.parquet,True,1418,12,0.013648,Operational
3,session_result.parquet,True,1374,11,0.020679,Operational
4,laps.parquet,True,63676,17,2.357738,Operational
5,weather.parquet,True,9485,10,0.147492,Operational
6,stints.parquet,True,3254,8,0.015198,Operational
7,starting_grid.parquet,True,1375,6,0.011728,Operational
8,intervals.parquet,True,1392480,8,22.333760,Operational
9,position.parquet,True,27767,5,0.224882,Operational


In [3]:
fig = px.bar(
    file_df,
    x="file",
    y="rows",
    color="endpoint_group",
    log_y=True,
    title="Silver File Coverage: Row Counts by Endpoint",
    labels={"rows": "Rows (log scale)", "file": "Silver artifact"},
)
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "file_row_coverage.html", include_plotlyjs="cdn")
fig.show()

## Race and Sprint Scope

OpenF1 encodes Sprint race rows with `session_type = Race` and `session_name = Sprint`. Silver therefore creates a normalized `event_type` so feature engineering can distinguish Grand Prix races from Sprint races.

In [4]:
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")
sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)
session_distribution = (
    sessions.groupby(["year", "event_type"], dropna=False)
    .size()
    .reset_index(name="sessions")
    .sort_values(["year", "event_type"])
)
session_distribution.to_csv(OUTPUT_TABLES / "session_distribution.csv", index=False)
display(session_distribution)

,year,event_type,sessions
0,2024,GRAND_PRIX_RACE,24
1,2024,SPRINT_RACE,6
2,2025,GRAND_PRIX_RACE,24
3,2025,SPRINT_RACE,6
4,2026,GRAND_PRIX_RACE,7
5,2026,SPRINT_RACE,3


In [5]:
fig = px.bar(
    session_distribution,
    x="year",
    y="sessions",
    color="event_type",
    barmode="group",
    title="Silver Session Scope: Grand Prix Race vs Sprint Race",
)
fig.write_html(OUTPUT_CHARTS / "session_distribution.html", include_plotlyjs="cdn")
fig.show()

## Critical Null Review

This check focuses on operational tables used immediately in Gold. Structural nulls from Bronze are allowed, but technical keys and core ML joins must remain populated.

In [6]:
critical_columns = {
    "sessions": ["session_key", "meeting_key", "session_name"],
    "drivers": ["session_key", "driver_number", "full_name"],
    "session_result": ["session_key", "driver_number", "position"],
    "laps": ["session_key", "driver_number", "lap_number", "lap_duration"],
    "weather": ["session_key", "date", "track_temperature", "air_temperature"],
    "starting_grid": ["session_key", "driver_number", "position"],
}
null_records = []
for table, columns in critical_columns.items():
    df = pd.read_parquet(CLEANED_DATA_PATH / f"{table}.parquet")
    for column in columns:
        if column not in df.columns:
            null_records.append({"table": table, "column": column, "rows": len(df), "null_count": len(df), "null_pct": 100.0, "status": "FAIL_MISSING_COLUMN"})
            continue
        null_count = int(df[column].isna().sum())
        null_records.append({
            "table": table,
            "column": column,
            "rows": len(df),
            "null_count": null_count,
            "null_pct": null_count / len(df) * 100 if len(df) else 0.0,
            "status": "PASS" if null_count == 0 else "REVIEW",
        })
null_df = pd.DataFrame(null_records)
null_df.to_csv(OUTPUT_TABLES / "critical_null_review.csv", index=False)
display(null_df.sort_values(["status", "null_pct"], ascending=[True, False]))

,table,column,rows,null_count,null_pct,status
0,sessions,session_key,70,0,0.00000,PASS
1,sessions,meeting_key,70,0,0.00000,PASS
2,sessions,session_name,70,0,0.00000,PASS
3,drivers,session_key,1418,0,0.00000,PASS
4,drivers,driver_number,1418,0,0.00000,PASS
5,drivers,full_name,1418,0,0.00000,PASS
6,session_result,session_key,1374,0,0.00000,PASS
7,session_result,driver_number,1374,0,0.00000,PASS
9,laps,session_key,63676,0,0.00000,PASS
10,laps,driver_number,63676,0,0.00000,PASS


In [7]:
plot_df = null_df[null_df["null_count"] > 0]
fig = px.bar(
    plot_df,
    x="null_pct",
    y="table",
    color="column",
    orientation="h",
    title="Silver Critical Null Review",
)
fig.write_html(OUTPUT_CHARTS / "critical_null_review.html", include_plotlyjs="cdn")
fig.show()

## Predictive Readiness Signals

The next checks validate the main race modeling signals: finish position distribution, lap-duration behavior, weather context, and grid-to-finish relationship.

In [8]:
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
weather = pd.read_parquet(CLEANED_DATA_PATH / "weather.parquet")
starting_grid = pd.read_parquet(CLEANED_DATA_PATH / "starting_grid.parquet")

finish = session_result.copy()
finish["finish_position"] = pd.to_numeric(finish["position"], errors="coerce")
finish_distribution = finish["finish_position"].value_counts(dropna=False).sort_index().reset_index()
finish_distribution.columns = ["finish_position", "drivers"]
finish_distribution.to_csv(OUTPUT_TABLES / "finish_position_distribution.csv", index=False)
display(finish_distribution.head(25))

,finish_position,drivers
0,1.0,68
1,2.0,68
2,3.0,68
3,4.0,68
4,5.0,68
5,6.0,68
6,7.0,68
7,8.0,68
8,9.0,68
9,10.0,68


In [9]:
fig = px.bar(
    finish_distribution,
    x="finish_position",
    y="drivers",
    title="Finish Position Distribution",
    labels={"finish_position": "Finish position", "drivers": "Driver-session rows"},
)
fig.write_html(OUTPUT_CHARTS / "finish_position_distribution.html", include_plotlyjs="cdn")
fig.show()

In [10]:
lap_stats = laps["lap_duration"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).reset_index()
lap_stats.columns = ["metric", "lap_duration_seconds"]
lap_stats.to_csv(OUTPUT_TABLES / "lap_duration_stats.csv", index=False)
display(lap_stats)

,metric,lap_duration_seconds
0,count,63676.000000
1,mean,91.986239
2,std,15.496155
3,min,67.694000
4,1%,70.213750
5,5%,73.517000
6,50%,90.485000
7,95%,118.722250
8,99%,150.836250
9,max,240.005000


In [11]:
sample_laps = laps[["lap_duration"]].dropna()
if len(sample_laps) > 100_000:
    sample_laps = sample_laps.sample(100_000, random_state=42)
fig = px.histogram(sample_laps, x="lap_duration", nbins=80, title="Lap Duration Distribution After Silver Cleaning")
fig.write_html(OUTPUT_CHARTS / "lap_duration_distribution.html", include_plotlyjs="cdn")
fig.show()

In [12]:
weather_cols = [column for column in ["air_temperature", "track_temperature", "humidity", "pressure", "wind_speed"] if column in weather.columns]
weather_stats = weather[weather_cols].describe().round(2).reset_index().rename(columns={"index": "metric"})
weather_stats.to_csv(OUTPUT_TABLES / "weather_stats.csv", index=False)
display(weather_stats)

,metric,air_temperature,track_temperature,humidity,pressure,wind_speed
0,count,9485.00,9485.00,9485.00,9485.00,9485.00
1,mean,23.15,34.75,55.23,998.68,1.82
2,std,4.91,9.59,16.67,27.87,1.05
3,min,12.10,12.20,14.00,918.90,0.00
4,25%,19.00,27.30,45.00,993.00,1.10
5,50%,23.00,34.60,55.20,1009.90,1.60
6,75%,27.20,43.10,68.00,1016.40,2.40
7,max,34.10,54.60,93.00,1031.00,7.20


In [13]:
grid_finish = starting_grid.merge(
    session_result[["session_key", "driver_number", "position"]],
    on=["session_key", "driver_number"],
    suffixes=("_grid", "_finish"),
)
grid_finish["grid_position"] = pd.to_numeric(grid_finish["position_grid"], errors="coerce")
grid_finish["finish_position"] = pd.to_numeric(grid_finish["position_finish"], errors="coerce")
grid_finish = grid_finish.dropna(subset=["grid_position", "finish_position"])
grid_finish.to_csv(OUTPUT_TABLES / "grid_finish_sample.csv", index=False)
corr = float(grid_finish[["grid_position", "finish_position"]].corr().iloc[0, 1]) if len(grid_finish) > 1 else float("nan")
display(grid_finish[["session_key", "driver_number", "grid_position", "finish_position"]].head(20))

,session_key,driver_number,grid_position,finish_position
0,9472,1,1,1.0
1,9472,16,2,4.0
2,9472,63,3,5.0
3,9472,55,4,3.0
4,9472,11,5,2.0
5,9472,14,6,9.0
6,9472,4,7,6.0
7,9472,81,8,8.0
8,9472,44,9,7.0
9,9472,27,10,16.0


In [14]:
fig = px.scatter(
    grid_finish,
    x="grid_position",
    y="finish_position",
    title=f"Grid Position vs Finish Position (corr={corr:.3f})",
)
if len(grid_finish) > 2:
    fit = np.polyfit(grid_finish["grid_position"], grid_finish["finish_position"], deg=1)
    x_line = np.array([grid_finish["grid_position"].min(), grid_finish["grid_position"].max()])
    y_line = fit[0] * x_line + fit[1]
    fig.add_scatter(x=x_line, y=y_line, mode="lines", name="Linear fit")
fig.write_html(OUTPUT_CHARTS / "grid_vs_finish.html", include_plotlyjs="cdn")
fig.show()

## Final Silver Gate

The Silver gate passes when all expected files exist, no critical columns are missing, and cleaned sessions contain only Grand Prix race or Sprint race rows.

In [15]:
missing_files = file_df[~file_df["exists"]]
missing_columns = null_df[null_df["status"].eq("FAIL_MISSING_COLUMN")]
invalid_event_rows = sessions[~sessions["event_type"].isin(["GRAND_PRIX_RACE", "SPRINT_RACE"])]
review_nulls = null_df[null_df["status"].eq("REVIEW")]

status = "PASS" if missing_files.empty and missing_columns.empty and invalid_event_rows.empty else "FAIL"
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": status,
    "total_files": int(len(file_df)),
    "total_rows": int(file_df["rows"].sum()),
    "grand_prix_sessions": int((sessions["event_type"] == "GRAND_PRIX_RACE").sum()),
    "sprint_sessions": int((sessions["event_type"] == "SPRINT_RACE").sum()),
    "critical_null_review_count": int(len(review_nulls)),
    "grid_finish_correlation": corr,
    "missing_files": missing_files.to_dict("records"),
    "missing_columns": missing_columns.to_dict("records"),
}
write_report("verify_cleaned_data", report)
write_insight(
    "Silver Verification Insights",
    [
        f"Verified {len(file_df)} Silver Parquet artifacts.",
        f"Grand Prix races: {report['grand_prix_sessions']}; Sprint races: {report['sprint_sessions']}.",
        f"Grid-to-finish correlation: {corr:.3f}.",
    ],
    [f"{row.table}.{row.column}: {row.null_count} nulls" for row in review_nulls.itertuples()],
    [
        "Use event_type rather than session_type to distinguish Grand Prix races from Sprint races.",
        "Treat remaining critical null reviews as feature-level decisions, not file integrity failures.",
        "Proceed to Gold only after this Silver verification status remains PASS.",
    ],
)
if status == "PASS":
    (CHECKPOINTS / "silver_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '01_verify_cleaned_data', 'timestamp': '2026-06-02T00:45:52.011623', 'status': 'PASS', 'total_files': 13, 'total_rows': 66416469, 'grand_prix_sessions': 55, 'sprint_sessions': 15, 'critical_null_review_count': 1, 'grid_finish_correlation': 0.7415668342234872, 'missing_files': [], 'missing_columns': []}
